# Resource Allocation Evaluation
Business Process Optimization

> This notebook evaluates resource allocation methods on the same simulator under controlled settings. The only intended experimental change is the allocation method itself.

In [1]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment check")
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")
print(f"NumPy    : {np.__version__}")
print(f"Pandas   : {pd.__version__}")
print(f"Seaborn  : {sns.__version__}")
print(f"Repo root guess: {Path.cwd()}")

Environment check
Python   : 3.11.6
Platform : macOS-26.3-arm64-arm-64bit
NumPy    : 2.3.5
Pandas   : 2.3.3
Seaborn  : 0.13.2
Repo root guess: /Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/simulation_evaluation/resource_allocation


## Evaluation Scope and Metrics

| # | Metric | Definition |
|---|--------|------------|
| 1 | **Average Cycle Time** | Mean case duration from first to last event (completed-case proxy documented later). |
| 2 | **Average Resource Occupation** | Mean ratio of working time to scheduled available time per resource. |
| 3 | **Resource Fairness (MAD / weighted MAD)** | Deviation of resource occupations from their mean; lower values indicate more balanced workload distribution. |

## Reproducibility and Notebook Configuration
This section fixes random seeds and visual/display settings to keep results consistent across runs and report exports.

In [2]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.options.display.float_format = "{:.4f}".format

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "resources").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
STUDY_DIR = REPO_ROOT / "simulation_evaluation" / "resource_allocation"
RESULTS_DIR = STUDY_DIR

print(f"Random seed fixed to: {RANDOM_SEED}")
print(f"Study directory      : {STUDY_DIR}")
print(f"Repository root      : {REPO_ROOT}")

Random seed fixed to: 42
Study directory      : /Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/simulation_evaluation/resource_allocation
Repository root      : /Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork


## Input Data and Replication Design

Simulation logs are stored under `data/<method>/run_NN.csv`.  
Each file is one independent replication of the same process model with the same demand profile — only the allocation method differs.

| Method label | Allocation strategy |
|---|---|
| `random` | Uniform random selection from eligible resources |
| `round_robin` | Strict round-robin over eligible resources |
| `shortest_queue` | Resource with the earliest next availability |
| `batch_k5` | Batching (k = 5) — releases a batch of 5 cases to the shortest-queue resource |

**Completed-case proxy:** cases with more than 2 recorded events (consistent with project convention).  
**Availability file:** `resources/availabilities/availabilities_advanced.csv` — used to compute scheduled working seconds per resource.


In [3]:
DATA_ROOT = STUDY_DIR / "data"

METHODS = {
    "random":         DATA_ROOT / "random",
    "round_robin":    DATA_ROOT / "round_robin",
    "shortest_queue": DATA_ROOT / "shortest_queue",
    "batch_k5":       DATA_ROOT / "batch_k5",
}

AVAIL_FILE = REPO_ROOT / "resources" / "availabilities" / "availabilities_advanced.csv"
REQUIRED_COLS = {"case:concept:name", "concept:name", "time:timestamp", "lifecycle:transition", "org:resource"}
MIN_EVENTS_PER_CASE = 2  # proxy for completed cases

# Sanity check
for label, path in METHODS.items():
    status = "✓" if path.exists() else "✗ missing"
    print(f"  {label:<16} {status}  ({path})")
print(f"\n  Availability file  {'✓' if AVAIL_FILE.exists() else '✗ missing'}  ({AVAIL_FILE})")

  random           ✓  (/Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/simulation_evaluation/resource_allocation/data/random)
  round_robin      ✓  (/Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/simulation_evaluation/resource_allocation/data/round_robin)
  shortest_queue   ✓  (/Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/simulation_evaluation/resource_allocation/data/shortest_queue)
  batch_k5         ✓  (/Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/simulation_evaluation/resource_allocation/data/batch_k5)

  Availability file  ✓  (/Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/resources/availabilities/availabilities_advanced.csv)


## Data Loading

Reads all replication CSV files for each method. Each file is validated for required columns and filtered to completed cases (proxy: > 2 events per case). Results are stored in `logs` — a dict mapping method label → list of DataFrames, one per replication.


In [10]:
def load_method_runs(method_dir: Path) -> list[pd.DataFrame]:
    """Load and validate all run_*.csv files in a method directory."""
    files = sorted(method_dir.glob("run_*.csv"))
    if not files:
        return []
    runs = []
    for f in files:
        df = pd.read_csv(f, parse_dates=["time:timestamp"])
        missing = REQUIRED_COLS - set(df.columns)
        if missing:
            raise ValueError(f"{f.name}: missing columns {missing}")
        # Filter to completed cases
        counts = df.groupby("case:concept:name").size()
        completed = counts[counts > MIN_EVENTS_PER_CASE].index
        df = df[df["case:concept:name"].isin(completed)].copy()
        runs.append(df)
    return runs


logs: dict[str, list[pd.DataFrame]] = {}
for method, path in METHODS.items():
    if not path.exists():
        print(f"  {method:<16} — directory not found, skipping")
        logs[method] = []
        continue
    runs = load_method_runs(path)
    logs[method] = runs
    total_cases = sum(df["case:concept:name"].nunique() for df in runs)
    print(f"  {method:<16} {len(runs):>2} run(s)  |  {total_cases:>6} completed cases total")


  random            5 run(s)  |    7614 completed cases total
  round_robin       5 run(s)  |    7562 completed cases total
  shortest_queue    5 run(s)  |    7496 completed cases total
  batch_k5          5 run(s)  |    7149 completed cases total


## Metric Helper Functions

Shared utilities for all three metrics. Implementations mirror `evaluation.ipynb` for consistency.


In [11]:
import datetime

# Activities with explicit start/complete lifecycle pairs (resource time is measurable)
W_ACTIVITIES = [
    "W_Call after offers",
    "W_Call incomplete files",
    "W_Complete application",
    "W_Handle leads",
    "W_Validate application",
]

DAY_ZERO = datetime.date(2016, 1, 1)  # DayId=0 anchor (matches availability schedule)


def compute_cycle_times(df: pd.DataFrame) -> pd.Series:
    """Per-case cycle time in hours (completed cases only)."""
    ct = df.groupby("case:concept:name")["time:timestamp"].agg(["min", "max"])
    return (ct["max"] - ct["min"]).dt.total_seconds() / 3600


def compute_working_seconds(df: pd.DataFrame) -> pd.Series:
    """Total working seconds per resource from start/complete event pairs."""
    w = df[df["concept:name"].isin(W_ACTIVITIES)].copy()
    starts = (
        w[w["lifecycle:transition"] == "start"]
        [["case:concept:name", "concept:name", "org:resource", "time:timestamp"]]
        .rename(columns={"time:timestamp": "start_time"})
    )
    completes = (
        w[w["lifecycle:transition"] == "complete"]
        [["case:concept:name", "concept:name", "org:resource", "time:timestamp"]]
        .rename(columns={"time:timestamp": "end_time"})
    )
    merged = pd.merge(starts, completes, on=["case:concept:name", "concept:name", "org:resource"])
    merged["working_seconds"] = (merged["end_time"] - merged["start_time"]).dt.total_seconds().clip(lower=0)
    return merged.groupby("org:resource")["working_seconds"].sum()


def compute_available_seconds(avail_df: pd.DataFrame, start_date: datetime.date, end_date: datetime.date) -> pd.Series:
    """Total scheduled seconds per resource over the simulation window."""
    result = {}
    for resource, grp in avail_df.groupby("Resource"):
        total = 0.0
        day = start_date
        while day <= end_date:
            day_id = (day - DAY_ZERO).days % 7
            row = grp[grp["DayId"] == day_id]
            if not row.empty:
                r = row.iloc[0]
                shift_s = datetime.datetime.combine(day, r["StartTime"])
                shift_e = datetime.datetime.combine(day, r["EndTime"])
                total += max(0.0, (shift_e - shift_s).total_seconds() - r["DurationMin"] * 60)
            day += datetime.timedelta(days=1)
        result[resource] = total
    return pd.Series(result, name="available_seconds")


def compute_occupation(working_s: pd.Series, available_s: pd.Series) -> pd.DataFrame:
    """DataFrame with working_seconds, available_seconds, and occupation per resource."""
    df = pd.DataFrame({"working_seconds": working_s, "available_seconds": available_s})
    df["working_seconds"] = df["working_seconds"].fillna(0)
    df = df[df["available_seconds"] > 0].copy()
    df["occupation"] = (df["working_seconds"] / df["available_seconds"]).clip(0, 1)
    return df


def fairness_metrics(occ_series: pd.Series, avail_seconds: pd.Series | None = None) -> tuple:
    """(mean_occ, MAD, weighted MAD). Lower MAD = more balanced workload."""
    mean_occ = occ_series.mean()
    mad = (occ_series - mean_occ).abs().mean()
    if avail_seconds is not None:
        w = avail_seconds / avail_seconds.sum()
        wmad = (w * (occ_series - mean_occ).abs()).sum()
    else:
        wmad = np.nan
    return mean_occ, mad, wmad


# Load and parse availability schedule once
avail_df = pd.read_csv(AVAIL_FILE)
avail_df["StartTime"]   = pd.to_datetime(avail_df["StartTime"],   format="%H:%M:%S").dt.time
avail_df["EndTime"]     = pd.to_datetime(avail_df["EndTime"],     format="%H:%M:%S").dt.time
avail_df["DurationMin"] = avail_df["DurationMin"].fillna(0).astype(int)

print(f"Helpers defined. Availability schedule: {avail_df['Resource'].nunique()} resources, {len(avail_df)} shift rows.")


Helpers defined. Availability schedule: 140 resources, 763 shift rows.


## Per-Method Metric Aggregation

Iterates over every replication of each method and computes:
- mean cycle time (hours)
- mean resource occupation
- workload fairness MAD and weighted MAD

Results are collected into a single tidy DataFrame `summary_df` (one row per method × replication).


In [12]:
records = []

for method, runs in logs.items():
    if not runs:
        continue
    for run_idx, df in enumerate(runs):
        # Derive simulation window from the log itself
        ts = df["time:timestamp"]
        sim_start = ts.min().date()
        sim_end = ts.max().date()

        # Metric 1: cycle time
        ct = compute_cycle_times(df)
        mean_ct = ct.mean()

        # Metric 2 & 3: occupation + fairness
        working_s = compute_working_seconds(df)
        available_s = compute_available_seconds(avail_df, sim_start, sim_end)
        occ_df = compute_occupation(working_s, available_s)
        mean_occ, mad, wmad = fairness_metrics(occ_df["occupation"], occ_df["available_seconds"])

        records.append({
            "method": method,
            "run": run_idx,
            "n_cases": df["case:concept:name"].nunique(),
            "mean_ct_h": mean_ct,
            "mean_occ": mean_occ,
            "fairness_mad": mad,
            "fairness_wmad": wmad,
        })

if records:
    summary_df = pd.DataFrame(records)
    print(summary_df.to_string(index=False))
else:
    summary_df = pd.DataFrame(columns=["method", "run", "n_cases", "mean_ct_h", "mean_occ", "fairness_mad", "fairness_wmad"])
    print("No data loaded — run the study script and place CSVs in data/<method>/")

        method  run  n_cases  mean_ct_h  mean_occ  fairness_mad  fairness_wmad
        random    0     1582     8.1578    0.2065        0.1031         0.1084
        random    1     1538     7.9702    0.1942        0.0907         0.0924
        random    2     1533     8.4024    0.2096        0.1102         0.1137
        random    3     1443     8.6698    0.1836        0.0928         0.0980
        random    4     1518     8.0664    0.2104        0.1108         0.1172
   round_robin    0     1534     8.9448    0.2283        0.1621         0.1672
   round_robin    1     1507     8.9737    0.2216        0.1609         0.1683
   round_robin    2     1528     8.9327    0.2396        0.1771         0.1838
   round_robin    3     1495     8.6329    0.2350        0.1693         0.1750
   round_robin    4     1498     8.4060    0.2038        0.1458         0.1523
shortest_queue    0     1497    32.3432    0.2515        0.3043         0.2990
shortest_queue    1     1545    48.9387    0.2280   